# Churn Prediction & Retention Strategy for Company A
## GCI World 2026 April — Final Assignment
**Author:** GermanRojas45  ·  Germán Rojas Serrano  
**Date:** May 2026  
**Course:** GCI World 2026 April — Matsuo-Iwasawa Laboratory, The University of Tokyo


## 1. Setup & Data Loading


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, accuracy_score,
                             classification_report, confusion_matrix, roc_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Plot style
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Blues_d')

print('Libraries loaded successfully.')

In [ ]:
# Mount Google Drive (adjust path as needed)
from google.colab import drive
drive.mount('/content/drive')

PATH = '/content/drive/MyDrive/Final Assignment/telecom/'

client = pd.read_csv(PATH + 'Client.csv')
record = pd.read_csv(PATH + 'Record.csv')

print(f'Client.csv  : {client.shape[0]:,} rows × {client.shape[1]} columns')
print(f'Record.csv  : {record.shape[0]:,} rows × {record.shape[1]} columns')

## 2. Data Overview & Merging


In [ ]:
# Merge datasets on Customer_ID
df = pd.merge(record, client, on='Customer_ID', how='inner')
print(f'Merged dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

In [ ]:
# Missing values overview
missing = (df.isnull().sum() / len(df) * 100).round(2)
missing = missing[missing > 0].sort_values(ascending=False)
print('Columns with missing values (% missing):')
print(missing.to_string())

## 3. Exploratory Data Analysis (EDA)


In [ ]:
# === EDA 1: Churn Distribution ===
churn_counts = df['churn'].value_counts()
churn_rate = df['churn'].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Retained (0)', 'Churned (1)'], churn_counts.values,
            color=['#065A82', '#E85D04'], edgecolor='white', linewidth=0.5)
axes[0].set_title('Churn Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 300, f'{v:,}\n({v/len(df)*100:.1f}%)',
                 ha='center', fontsize=11, fontweight='bold')

# Pie chart
axes[1].pie(churn_counts.values, labels=['Retained', 'Churned'],
            colors=['#065A82', '#E85D04'],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Churn Rate Overview', fontsize=14, fontweight='bold')

plt.suptitle(f'Overall Churn Rate: {churn_rate:.1%}', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig1_churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total customers: {len(df):,}')
print(f'Churn rate: {churn_rate:.2%}')

In [ ]:
# === EDA 2: Churn by Equipment Age (eqpdays) ===
df['eqp_q'] = pd.qcut(df['eqpdays'], q=4,
                       labels=['Q1\nNewest (<146d)', 'Q2\n(146-363d)',
                               'Q3\n(363-624d)', 'Q4\nOldest (>624d)'])
eqp_churn = df.groupby('eqp_q', observed=True)['churn'].mean() * 100

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(eqp_churn.index, eqp_churn.values,
              color=['#CAE9FF', '#7EC8E3', '#0892D0', '#065A82'], edgecolor='white')
ax.set_title('Churn Rate by Equipment Age (Quartiles)', fontsize=14, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_xlabel('Equipment Age Quartile')
ax.axhline(y=df['churn'].mean()*100, color='#E85D04', linestyle='--', linewidth=1.5, label='Overall average')
ax.legend()
for bar, val in zip(bars, eqp_churn.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 70)
plt.tight_layout()
plt.savefig('fig2_churn_by_eqpdays.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === EDA 3: Churn by Usage Change (change_mou) ===
df['mou_q'] = pd.qcut(df['change_mou'], q=4,
                       labels=['Q1\nDeclining Most', 'Q2\nSlightly Declining',
                               'Q3\nSlight Growth', 'Q4\nGrowing Most'])
mou_churn = df.groupby('mou_q', observed=True)['churn'].mean() * 100

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(mou_churn.index, mou_churn.values,
              color=['#065A82', '#0892D0', '#7EC8E3', '#CAE9FF'], edgecolor='white')
ax.set_title('Churn Rate by Usage Trend (change_mou)', fontsize=14, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_xlabel('Monthly Usage Change Quartile')
ax.axhline(y=df['churn'].mean()*100, color='#E85D04', linestyle='--', linewidth=1.5, label='Overall average')
ax.legend()
for bar, val in zip(bars, mou_churn.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 70)
plt.tight_layout()
plt.savefig('fig3_churn_by_mou_change.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === EDA 4: Churn by Customer Tenure ===
df['tenure_seg'] = pd.cut(df['months'], bins=[0, 12, 24, 36, 60, 200],
                           labels=['<1 yr', '1-2 yr', '2-3 yr', '3-5 yr', '5+ yr'])
tenure_churn = df.groupby('tenure_seg', observed=True)['churn'].mean() * 100

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#065A82', '#0892D0', '#1C7293', '#7EC8E3', '#CAE9FF']
bars = ax.bar(tenure_churn.index, tenure_churn.values, color=colors, edgecolor='white')
ax.set_title('Churn Rate by Customer Tenure', fontsize=14, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_xlabel('Tenure Segment')
ax.axhline(y=df['churn'].mean()*100, color='#E85D04', linestyle='--', linewidth=1.5, label='Overall average')
ax.legend()
for bar, val in zip(bars, tenure_churn.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 70)
plt.tight_layout()
plt.savefig('fig4_churn_by_tenure.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Engineering


In [ ]:
# === Feature Engineering ===
# 1. High-risk flag: old device AND declining usage
df['high_risk'] = ((df['eqpdays'] > df['eqpdays'].quantile(0.75)) &
                   (df['change_mou'] < df['change_mou'].quantile(0.25))).astype(int)

# 2. Call completion rate
df['completion_rate'] = df['comp_vce_Mean'] / (df['attempt_Mean'] + 1e-5)

# 3. Revenue per minute
df['rev_per_mou'] = df['rev_Mean'] / (df['mou_Mean'] + 1e-5)

# 4. Overage ratio
df['overage_ratio'] = df['ovrmou_Mean'] / (df['mou_Mean'] + 1e-5)

print('High-risk segment statistics:')
hr = df[df['high_risk'] == 1]
print(f'  Customers in high-risk segment: {len(hr):,} ({len(hr)/len(df)*100:.1f}% of base)')
print(f'  Churn rate (high-risk):         {hr["churn"].mean():.1%}')
print(f'  Churn rate (other):             {df[df["high_risk"]==0]["churn"].mean():.1%}')

## 5. Model Building


In [ ]:
# === Feature Selection ===
numeric_features = [
    'rev_Mean','mou_Mean','totmrc_Mean','da_Mean','ovrmou_Mean','ovrrev_Mean',
    'change_mou','change_rev','drop_vce_Mean','drop_dat_Mean','blck_vce_Mean',
    'custcare_Mean','ccrndmou_Mean','inonemin_Mean','roam_Mean',
    'mou_cvce_Mean','peak_vce_Mean','opk_vce_Mean','drop_blk_Mean',
    'attempt_Mean','complete_Mean','callfwdv_Mean','callwait_Mean',
    'months','totcalls','totmou','totrev','avgrev','avgmou','eqpdays',
    'avg3mou','avg3qty','avg3rev','avg6mou','avg6qty','avg6rev',
    'uniqsubs','actvsubs','hnd_price','phones','models','lor','adults',
    'income','numbcars','forgntvl',
    'completion_rate','rev_per_mou','overage_ratio','high_risk'
]

features = [c for c in numeric_features if c in df.columns]
X = df[features].copy()
y = df['churn'].copy()

# Impute missing values
imputer = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=features)

# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_imp, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training set:  {X_train.shape[0]:,} samples')
print(f'Test set:      {X_test.shape[0]:,} samples')
print(f'Features used: {len(features)}')

In [ ]:
# === Model 1: Logistic Regression ===
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_sc, y_train)

lr_prob = lr.predict_proba(X_test_sc)[:, 1]
lr_pred = lr.predict(X_test_sc)
lr_auc  = roc_auc_score(y_test, lr_prob)
lr_acc  = accuracy_score(y_test, lr_pred)

print('=== Logistic Regression ===')
print(f'Model Name    : Logistic Regression (L2, balanced class weights)')
print(f'Evaluation    : AUC-ROC')
print(f'AUC-ROC Score : {lr_auc:.4f}')
print(f'Accuracy      : {lr_acc:.4f}')
print()
print(classification_report(y_test, lr_pred, target_names=['Retained','Churned']))

In [ ]:
# === Model 2: Random Forest ===
rf = RandomForestClassifier(
    n_estimators=100, max_depth=10,
    random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(X_train, y_train)

rf_prob = rf.predict_proba(X_test)[:, 1]
rf_pred = rf.predict(X_test)
rf_auc  = roc_auc_score(y_test, rf_prob)
rf_acc  = accuracy_score(y_test, rf_pred)

print('=== Random Forest ===')
print(f'Model Name    : Random Forest Classifier (100 trees, max_depth=10, balanced)')
print(f'Evaluation    : AUC-ROC')
print(f'AUC-ROC Score : {rf_auc:.4f}')
print(f'Accuracy      : {rf_acc:.4f}')
print()
print(classification_report(y_test, rf_pred, target_names=['Retained','Churned']))

In [ ]:
# === Model Comparison: ROC Curves ===
fig, ax = plt.subplots(figsize=(8, 6))

for model_name, prob in [('Logistic Regression', lr_prob), ('Random Forest', rf_prob)]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, linewidth=2, label=f'{model_name} (AUC = {auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUC = 0.5)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig('fig5_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Feature Importance (Random Forest) ===
importance = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)
top15 = importance.tail(15)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#065A82' if f in ['eqpdays','months','change_mou','hnd_price','mou_Mean']
          else '#7EC8E3' for f in top15.index]
top15.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Top 15 Feature Importances (Random Forest)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')

highlight = mpatches.Patch(color='#065A82', label='Key churn predictors')
other = mpatches.Patch(color='#7EC8E3', label='Supporting features')
ax.legend(handles=[highlight, other])
plt.tight_layout()
plt.savefig('fig6_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top 5 predictors:')
print(importance.tail(5).sort_values(ascending=False).round(4).to_string())

## 6. Business Proposal & Impact Quantification


In [ ]:
# === Business Impact Calculation ===
total_customers   = len(df)
churned_customers = df['churn'].sum()
churn_rate        = df['churn'].mean()
avg_monthly_rev   = df['avgrev'].mean()
annual_ltv        = avg_monthly_rev * 12

# Revenue at risk
annual_revenue_at_risk = churned_customers * avg_monthly_rev * 12

# High-risk segment (EDA-defined)
high_risk_count    = df['high_risk'].sum()
high_risk_churn    = df[df['high_risk']==1]['churn'].mean()

# Apply Random Forest at threshold=0.5 to the high-risk segment
# (AUC-ROC measures ranking quality; actual flagging uses a probability threshold)
all_probs = rf.predict_proba(X_imp)[:,1]
df_temp = df.copy()
df_temp['churn_prob'] = all_probs
flagged_high_risk = int(((df_temp['churn_prob'] >= 0.5) & (df_temp['high_risk'] == 1)).sum())

# Conservative intervention: 25% of flagged high-risk customers retained
intervention_save_rate  = 0.25
retention_program_cost  = 20   # $20/customer/year (discount + outreach)

customers_saved     = int(flagged_high_risk * high_risk_churn * intervention_save_rate)
revenue_saved       = customers_saved * avg_monthly_rev * 12
program_cost        = flagged_high_risk * retention_program_cost
net_benefit         = revenue_saved - program_cost
roi                 = (net_benefit / program_cost) * 100

print('=== BUSINESS IMPACT SUMMARY ===')
print(f'Total customers:               {total_customers:>10,}')
print(f'Churned customers:             {churned_customers:>10,}')
print(f'Current churn rate:            {churn_rate:>10.1%}')
print()
print('Note: churn = 1 if customer churned within 31-60 days after observation.')
print('This is a short-term signal window, not calendar-year attrition.')
print()
print(f'Avg monthly revenue/customer:  ${avg_monthly_rev:>9.2f}')
print(f'Annual revenue at risk:        ${annual_revenue_at_risk:>9,.0f}')
print()
print('--- Targeted Retention Program ---')
print(f'High-risk segment (EDA):       {high_risk_count:>10,}  (60.4% churn rate)')
print(f'Flagged by model (thr=0.5):    {flagged_high_risk:>10,}')
print(f'Customers saved (est.):        {customers_saved:>10,}  (25% intervention success)')
print(f'Revenue saved (annual):        ${revenue_saved:>9,.0f}')
print(f'Program cost (annual):         ${program_cost:>9,.0f}')
print(f'Net benefit (annual):          ${net_benefit:>9,.0f}')
print(f'ROI:                           {roi:>9.0f}%')

In [ ]:
# === Summary: Model Performance ===
print('=' * 50)
print('FINAL MODEL COMPARISON SUMMARY')
print('=' * 50)
print(f'{"Model":<28} {"AUC-ROC":>8} {"Accuracy":>10}')
print('-' * 50)
print(f'{"Logistic Regression":<28} {lr_auc:>8.4f} {lr_acc:>10.4f}')
print(f'{"Random Forest":<28} {rf_auc:>8.4f} {rf_acc:>10.4f}')
print('=' * 50)
print()
print('SELECTED MODEL: Random Forest')
print(f'  → AUC-ROC: {rf_auc:.4f}  |  Accuracy: {rf_acc:.4f}')
print()
print('KEY FINDINGS:')
print('  1. Equipment age (eqpdays) is the strongest churn predictor')
print('  2. Declining usage (change_mou) is the 2nd strongest predictor')
print('  3. High-risk segment (old device + declining usage):')
print(f'     {high_risk_count:,} customers, {high_risk_churn:.1%} churn rate')
print()
print('BUSINESS PROPOSAL:')
print('  → Proactive Retention Program targeting high-risk customers')
print(f'  → Estimated annual revenue saved: ${revenue_saved:,.0f}')
print(f'  → ROI: {roi:.0f}%')